# Section 9: Give Dock a voice

Goal: drivers leave voicemails. Dock transcribes them with Voxtral, extracts the shipment and the issue,
answers by voice, and we measure how long each stage takes.

## Step 1. The voicemails

In [1]:
import json, os, re, wave
from pathlib import Path
from dotenv import load_dotenv
from mistralai.client import Mistral
from mistralai.client.utils import BackoffStrategy, RetryConfig

load_dotenv()
retry = RetryConfig("backoff", BackoffStrategy(1000, 20000, 2.0, 120000),
                    retry_connection_errors=True)
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"], retry_config=retry,
                 timeout_ms=120000)

CALLS = sorted(Path("dock/calls").glob("*.wav"))
truth = json.loads(open("dock/calls/truth.json").read())
for path in CALLS:
    w = wave.open(str(path))
    seconds = w.getnframes() / w.getframerate()
    t = truth[path.stem]
    print(path.stem, f"{seconds:4.1f} s", t["shipment_id"], t["issue"], "callback" if t["callback"] else "")

C1 14.5 s HF-26038 delay callback
C2 11.8 s HF-26016 question callback
C3 12.8 s HF-26025 arrived 
C4  7.0 s HF-26019 address callback
C5 10.0 s HF-26036 damage callback
C6  9.2 s HF-26040 delay 


## Step 2. Transcribe one call

In [2]:
import time

def transcribe(path, model="voxtral-mini-latest", bias=None):
    start = time.time()
    r = client.audio.transcriptions.complete(
        model=model, language="en", context_bias=bias,
        file={"file_name": path.name, "content": path.read_bytes()})
    return r.text, time.time() - start

text, seconds = transcribe(CALLS[0])
print(f"{seconds:.1f} s")
print(text)
print()
print(truth["C1"]["script"])

2.8 s
Hi, this is Marco, driving for Lotus. I'm stuck on the A7 near Hamburg. Big accident. Shipment HF26038 will be about two hours late. Call me back if that's a problem.

Hi, this is Marco, driving for Lotus. I'm stuck on the A7 near Hamburg, big accident. Shipment H F two six zero three eight will be about two hours late. Call me back if that's a problem.


## Step 3. Spoken numbers

In [3]:
transcripts = {}
for path in CALLS:
    text, seconds = transcribe(path)
    transcripts[path.stem] = text
    print(path.stem, f"{seconds:.1f} s |", text[:95])

NAMES = {"C1": "Marco", "C2": "Jenny", "C3": "Tom", "C4": "Pieter", "C5": "Sam", "C6": "Ana"}

def names_right(transcripts):
    return [c for c, name in NAMES.items() if re.search(rf"\b{name}\b", transcripts[c])]

C1 1.2 s | Hi, this is Marco, driving for Lotus. I'm stuck on the A7 near Hamburg. Big accident. Shipment 


C2 0.9 s | Morning, Jenny here. I'm at the Felixstowe Gate with HF26016. The gate says the booking referen


C3 2.0 s | Hello, it's Tom. Just letting you know container for HF26025 is delivered to the warehouse in L


C4 1.4 s | This is Peter's driver for shipment 26019. The address on my papers is an old warehouse. It's c


C5 1.1 s | Hi, Sam calling about HF26036. Two pallets fell over during unloading. Some cartons are crushed


C6 1.0 s | Yeah, hi, it's Anna. Running about 40 minutes behind with HF26040 traffic at the port. No reply


## Step 4. Extract, normalize, score

In [4]:
import re
from typing import Literal, Optional
from pydantic import BaseModel, Field, field_validator

class CallNote(BaseModel):
    shipment_id: Optional[str] = Field(description="The shipment number as said, like HF-26038")
    issue: Literal["delay", "arrived", "address", "damage", "question"] = Field(
        description="delay: running late · arrived: delivered · address: wrong or closed delivery "
                    "address · damage: goods damaged · question: anything else, like a gate or paperwork problem")
    callback: bool = Field(description="false only if the driver says no reply or call back is needed")

    @field_validator("shipment_id")
    @classmethod
    def normalize_ref(cls, value):
        if value is None:
            return None
        digits = re.sub(r"\D", "", value)
        return f"HF-{digits}" if len(digits) == 5 else value


def note(text, model="mistral-small-latest"):
    start = time.time()
    r = client.chat.parse(model=model, response_format=CallNote, messages=[
        {"role": "system", "content": "You write call notes for Harbor Freight's ops team."},
        {"role": "user", "content": text}])
    return r.choices[0].message.parsed, time.time() - start


def score(transcripts):
    right = {"shipment_id": 0, "issue": 0, "callback": 0}
    for cid, text in transcripts.items():
        n, _ = note(text)
        for f in right:
            ok = getattr(n, f) == truth[cid][f]
            right[f] += ok
            if not ok:
                print(" ", cid, f, repr(getattr(n, f)), "expected", repr(truth[cid][f]))
    return {f: f"{v}/{len(transcripts)}" for f, v in right.items()}

print(score(transcripts))

  C2 issue 'address' expected 'question'


{'shipment_id': '6/6', 'issue': '5/6', 'callback': '6/6'}


## Step 5. Tell Voxtral what words to expect

In [5]:
import sqlite3

db = sqlite3.connect("dock/dock.sqlite")
ACTIVE = [r[0] for r in db.execute("SELECT id FROM shipments WHERE status != 'delivered'")]
print("names right, no bias:", names_right(transcripts))

BIAS = list(NAMES.values()) + ["Felixstowe", "Lotus"] + ACTIVE    # one word or code per item
print(len(BIAS), "bias items ->", BIAS[:8], "...")

biased = {}
for path in CALLS:
    text, seconds = transcribe(path, bias=BIAS)
    biased[path.stem] = text
    print(path.stem, f"{seconds:.1f} s |", text[:95])
print("names right, with bias:", names_right(biased))
print(score(biased))

names right, no bias: ['C1', 'C2', 'C3', 'C5']
46 bias items -> ['Marco', 'Jenny', 'Tom', 'Pieter', 'Sam', 'Ana', 'Felixstowe', 'Lotus'] ...


C1 1.6 s | Hi, this is Marco, driving for Lotus. I'm stuck on the A7 near Hamburg. Big accident. Shipment 


C2 0.9 s | Morning, Jenny here. I'm at the Felixstowe Gate with HF 26016. The gate says the booking refere


C3 1.5 s | Hello, it's Tom. Just letting you know container for HF26025 is delivered to the warehouse in L


C4 0.9 s | This is Pieter's driver for shipment 26019. The address on my papers is an old warehouse. It's 


C5 1.3 s | Hi, Sam calling about HF26036. Two pallets fell over during unloading. Some cartons are crushed


C6 1.1 s | Yeah, hi, it's Ana. Running about 40 minutes behind with HF26040 traffic at the port. No reply 
names right, with bias: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6']


{'shipment_id': '6/6', 'issue': '6/6', 'callback': '6/6'}


## Step 6. Answer by voice

In [6]:
import base64

VOICES = {v.slug: v.id for v in client.audio.voices.list(limit=100).items}   # once, outside the timing

def speak(text, voice="gb_oliver_neutral", model="voxtral-mini-tts-latest"):
    start = time.time()
    r = client.audio.speech.complete(model=model, input=text, voice_id=VOICES[voice],
                                     response_format="wav")
    audio = base64.b64decode(r.audio_data)
    return audio, time.time() - start

n, _ = note(biased["C2"])
r = client.chat.complete(model="mistral-small-latest", messages=[
    {"role": "system", "content": "You are Dock. Reply to a driver's voicemail in two short sentences. "
                                  "Never promise a time you don't know."},
    {"role": "user", "content": biased["C2"]}])
answer = r.choices[0].message.content
audio, seconds = speak(answer)
Path("reply_C2.wav").write_bytes(audio)
print(answer)
print(f"speech: {seconds:.1f} s, {len(audio) // 1000} kB -> reply_C2.wav")

C:\Users\Admin\AppData\Local\Temp\ipykernel_8132\2468004023.py:3: DeprecationWarning: warning: ** DEPRECATED ** - Offset pagination will not be supported anymore. Use GET /v2/audio/voices instead.. Use search instead.
  VOICES = {v.slug: v.id for v in client.audio.voices.list(limit=100).items}   # once, outside the timing


Hey Jenny, thanks for calling. I'll get someone to check the booking reference and call you back shortly.
speech: 3.8 s, 245 kB -> reply_C2.wav


## Step 7. The latency budget

In [7]:
import statistics

stages = {"transcribe": [], "understand": [], "reply text": [], "speech": []}
for path in CALLS:
    text, s = transcribe(path, bias=BIAS)
    stages["transcribe"].append(s)
    _, s = note(text)
    stages["understand"].append(s)
    start = time.time()
    r = client.chat.complete(model="mistral-small-latest", messages=[
        {"role": "system", "content": "You are Dock. Reply to a driver's voicemail in two short sentences."},
        {"role": "user", "content": text}])
    stages["reply text"].append(time.time() - start)
    _, s = speak(r.choices[0].message.content)
    stages["speech"].append(s)

total = 0
for stage, times in stages.items():
    median = statistics.median(times)
    total += median
    print(f"{stage:11} median {median:4.1f} s   max {max(times):4.1f} s")
print(f"{'total':11} median {total:4.1f} s")

transcribe  median  1.0 s   max  1.2 s
understand  median  0.7 s   max  2.3 s
reply text  median  0.7 s   max  0.9 s
speech      median  4.2 s   max  7.2 s
total       median  6.7 s
